In [0]:
from pyspark.sql import functions as F 

SOURCE = "s3a://data5035-spring26/drone_data.csv"


In [0]:
# Radiation Sensor Data Modeling (Star Schema Design)

## Overview

The original dataset contains time-series radiation measurements collected from GPS-based sensor units. Each row includes:

- Timestamp
- Location (latitude/longitude)
- GPS unit metadata
- Multiple radiation detector readings (Gamma, Cesium-137, Thorium-232)
- Repeated calibration data for each device

This structure is **denormalized and inefficient**, as it repeats metadata across rows and limits analytical flexibility.

---

## Problem

The dataset suffers from:

- ❌ Repeated calibration metadata in every row  
- ❌ Wide-table design (multiple radiation columns)  
- ❌ Poor scalability for additional sensors  
- ❌ Limited query flexibility  

---

## Solution: Star Schema Design

We restructure the dataset into a **star schema** consisting of:

- ⭐ A central **fact table** for measurements  
- 📦 Multiple **dimension tables** for descriptive attributes  

---

## Fact Table

### `FACT_RADIATION_MEASUREMENT`

| Column              | Description                         |
|--------------------|--------------------------------------|
| measurement_id (PK)| Unique measurement identifier        |
| timestamp_id (FK)  | Reference to time dimension          |
| location_id (FK)   | Reference to location dimension      |
| gps_unit_id (FK)   | Reference to GPS unit                |
| detector_id (FK)   | Reference to detector                |
| radiation_type_id (FK) | Type of radiation measured       |
| level              | Radiation level value                |

---

## Key Transformation: UNPIVOT

The original dataset stores radiation types as separate columns:

- `GAMMA_LEVEL`
- `CESIUM_137_LEVEL`
- `THORIUM_232_LEVEL`

This is converted into a **long format**:

| radiation_type | level |
|----------------|-------|
| GAMMA          | 0.141 |
| CESIUM_137     | 0     |
| THORIUM_232    | 1.27  |

Each original row becomes **three rows** in the fact table.

---

## Dimension Tables

### 1. `DIM_TIME`

| Column         | Description        |
|----------------|--------------------|
| timestamp_id   | Primary key        |
| full_timestamp | Original timestamp |
| date           | Date portion       |
| hour           | Hour               |
| minute         | Minute             |
| second         | Second             |
| day_of_week    | Day name           |

---

### 2. `DIM_LOCATION`

| Column      | Description         |
|-------------|---------------------|
| location_id | Primary key         |
| gps_lat     | Latitude            |
| gps_lng     | Longitude           |

---

### 3. `DIM_GPS_UNIT`

| Column                 | Description              |
|------------------------|--------------------------|
| gps_unit_id            | Primary key              |
| unit_number            | Device identifier        |
| calibration_timestamp  | Last calibration time    |
| calibration_precision  | Precision value          |

---

### 4. `DIM_DETECTOR`

| Column                 | Description              |
|------------------------|--------------------------|
| detector_id            | Primary key              |
| detector_unit_number   | Detector identifier      |
| calibration_timestamp  | Last calibration time    |
| calibration_precision  | Precision value          |

---

### 5. `DIM_RADIATION_TYPE`

| Column             | Description         |
|--------------------|---------------------|
| radiation_type_id  | Primary key         |
| name               | Radiation type name |

Example values:

| radiation_type_id | name         |
|-------------------|--------------|
| 1                 | GAMMA        |
| 2                 | CESIUM_137   |
| 3                 | THORIUM_232  |

---

## Example Transformation

### Original Row

| timestamp | gamma | cesium | thorium |
|----------|------|--------|---------|
| 08:00    | 0.14 | 0      | 1.27    |

### Transformed Rows

| timestamp | radiation_type | level |
|----------|----------------|-------|
| 08:00    | GAMMA          | 0.14  |
| 08:00    | CESIUM_137     | 0     |
| 08:00    | THORIUM_232    | 1.27  |

---

## Benefits

- ✅ Eliminates redundant data  
- ✅ Improves storage efficiency  
- ✅ Enables flexible querying  
- ✅ Supports additional sensor types  
- ✅ Aligns with dimensional modeling best practices  

---

## Example Queries

### Average radiation by type

```sql
SELECT radiation_type_id, AVG(level)
FROM FACT_RADIATION_MEASUREMENT
GROUP BY radiation_type_id;




In [0]:
from pyspark.sql import functions as F 

SOURCE = "s3a://data5035-spring26/drone_data.csv" 
df = spark.read.option("header", True).csv(SOURCE)
display(df)

In [0]:
from pyspark.sql import functions as F

df = df.select(
    F.to_timestamp("COLLECTION_TIMESTAMP").alias("timestamp"),

    F.col("GPS_LAT").cast("double").alias("gps_lat"),
    F.col("GPS_LNG").cast("double").alias("gps_lng"),

    F.col("GPS_UNIT_NUMBER").alias("gps_unit"),
    F.to_timestamp("GPS_UNIT_CALIBRATION_TIMESTAMP").alias("gps_calibration_timestamp"),
    F.col("GPS_UNIT_CALIBRATION_PRECISION").cast("double").alias("gps_calibration_precision"),

    F.col("GAMMA_LEVEL").cast("double").alias("gamma_level"),
    F.col("GAMMA_DETECTOR_UNIT_NUMBER").alias("gamma_detector_unit"),
    F.to_timestamp("GAMMA_DETECTOR_CALIBRATION_TIMESTAMP").alias("gamma_calibration_timestamp"),
    F.col("GAMMA_DETECTOR_CALIBRATION_PRECISION").cast("double").alias("gamma_calibration_precision"),

    F.col("CESIUM_137_LEVEL").cast("double").alias("cesium_137_level"),
    F.col("CESIUM_137_DETECTOR_UNIT_NUMBER").alias("cesium_detector_unit"),
    F.to_timestamp("CESIUM_137_DETECTOR_CALIBRATION_TIMESTAMP").alias("cesium_calibration_timestamp"),
    F.col("CESIUM_137_DETECTOR_CALIBRATION_PRECISION").cast("double").alias("cesium_calibration_precision"),

    F.col("THORIUM_232_LEVEL").cast("double").alias("thorium_232_level"),
    F.col("THORIUM_232_DETECTOR_UNIT_NUMBER").alias("thorium_detector_unit"),
    F.to_timestamp("THORIUM_232_DETECTOR_CALIBRATION_TIMESTAMP").alias("thorium_calibration_timestamp"),
    F.col("THORIUM_232_DETECTOR_CALIBRATION_PRECISION").cast("double").alias("thorium_calibration_precision")
)
display(df)

In [0]:
df = df.withColumn("timestamp", F.to_timestamp("timestamp")) \
       .withColumn("gps_lat", F.col("gps_lat").cast("double")) \
       .withColumn("gps_lng", F.col("gps_lng").cast("double")) \
       .withColumn("gamma_level", F.col("gamma_level").cast("double")) \
       .withColumn("cesium_137_level", F.col("cesium_137_level").cast("double")) \
       .withColumn("thorium_232_level", F.col("thorium_232_level").cast("double"))

df.display()

In [0]:
dim_time = df.select("timestamp").distinct() \
    .withColumn("timestamp_id", F.monotonically_increasing_id()) \
    .withColumn("date", F.to_date("timestamp")) \
    .withColumn("hour", F.hour("timestamp")) \
    .withColumn("minute", F.minute("timestamp")) \
    .withColumn("second", F.second("timestamp")) \
    .withColumn("day_of_week", F.date_format("timestamp", "E"))

dim_time.display()

In [0]:
dim_location = df.select("gps_lat", "gps_lng").distinct() \
    .withColumn("location_id", F.monotonically_increasing_id())

dim_location.display()

In [0]:
dim_gps = df.select(
    F.col("gps_unit").alias("unit_number"),
    F.col("gps_calibration_timestamp").alias("calibration_timestamp"),
    F.col("gps_calibration_precision").alias("calibration_precision")
).distinct().withColumn("gps_unit_id", F.monotonically_increasing_id())

dim_gps.display()

In [0]:
gamma = df.select(
    F.col("gamma_detector_unit").alias("detector_unit_number"),
    F.col("gamma_calibration_timestamp").alias("calibration_timestamp"),
    F.col("gamma_calibration_precision").alias("calibration_precision")
)

cesium = df.select(
    F.col("cesium_detector_unit").alias("detector_unit_number"),
    F.col("cesium_calibration_timestamp").alias("calibration_timestamp"),
    F.col("cesium_calibration_precision").alias("calibration_precision")
)

thorium = df.select(
    F.col("thorium_detector_unit").alias("detector_unit_number"),
    F.col("thorium_calibration_timestamp").alias("calibration_timestamp"),
    F.col("thorium_calibration_precision").alias("calibration_precision")
)

dim_detector = gamma.union(cesium).union(thorium).distinct() \
    .withColumn("detector_id", F.monotonically_increasing_id())

dim_detector.display()

In [0]:
radiation_types = [("GAMMA",), ("CESIUM_137",), ("THORIUM_232",)]

dim_radiation = spark.createDataFrame(radiation_types, ["name"]) \
    .withColumn("radiation_type_id", F.monotonically_increasing_id())

dim_radiation.display()

In [0]:
df_long = df.select(
    "timestamp",
    "gps_lat",
    "gps_lng",
    "gps_unit",
 
    "gamma_detector_unit",
    "cesium_detector_unit",
    "thorium_detector_unit",

    F.expr("""
        stack(3,
            'GAMMA', gamma_level,
            'CESIUM_137', cesium_137_level,
            'THORIUM_232', thorium_232_level
        ) as (radiation_type, level)
    """)
)

df_long.display()

In [0]:
# Join time
fact = df_long.join(dim_time, on="timestamp", how="left")

# Join location
fact = fact.join(dim_location, on=["gps_lat", "gps_lng"], how="left")

# Join GPS unit
fact = fact.join(dim_gps, fact["gps_unit"] == dim_gps["unit_number"], "left")

# Join radiation type
fact = fact.join(dim_radiation, fact["radiation_type"] == dim_radiation["name"], "left")

display(fact)

In [0]:
fact = fact.withColumn(
    "detector_unit_number",
    F.when(F.col("radiation_type") == "GAMMA", F.col("gamma_detector_unit"))
     .when(F.col("radiation_type") == "CESIUM_137", F.col("cesium_detector_unit"))
     .when(F.col("radiation_type") == "THORIUM_232", F.col("thorium_detector_unit"))
)

fact = fact.join(dim_detector, on="detector_unit_number", how="left")
display(fact) 

In [0]:
fact_table = fact.select(
    F.monotonically_increasing_id().alias("measurement_id"),
    "timestamp_id",
    "location_id",
    "gps_unit_id",
    "detector_id",
    "radiation_type_id",
    "level"
)

fact_table.display()

In [0]:
dim_time.write.mode("overwrite").saveAsTable("dim_time")
dim_location.write.mode("overwrite").saveAsTable("dim_location")
dim_gps.write.mode("overwrite").saveAsTable("dim_gps_unit")
dim_detector.write.mode("overwrite").saveAsTable("dim_detector")
dim_radiation.write.mode("overwrite").saveAsTable("dim_radiation_type")
fact_table.write.mode("overwrite").saveAsTable("fact_radiation_measurement")

In [0]:
%md### Extra Credit: Type 2 Slowly Changing Dimensions

Type 2 Slowly Changing Dimensions were implemented for device-related dimensions whose descriptive attributes may change over time, specifically GPS units and radiation detectors. This preserves full historical context instead of overwriting prior calibration values.

For example, when a GPS unit receives a new calibration timestamp or calibration precision, the old dimension record is expired by setting its `effective_end_ts` and `is_current = false`, and a new row is inserted with the updated attributes, a new surrogate key, and `is_current = true`.

This approach allows each fact measurement to be linked to the exact version of the device metadata that was valid at the time of collection. As a result, historical analytics remain accurate even after recalibration events.


In [0]:
from pyspark.sql import functions as F

gps_src = (
    df.select(
        F.col("gps_unit").alias("unit_number"),
        F.col("gps_calibration_timestamp").alias("calibration_timestamp"),
        F.col("gps_calibration_precision").alias("calibration_precision")
    )
    .distinct()
    .withColumn("effective_start_ts", F.col("calibration_timestamp"))
    .withColumn("effective_end_ts", F.lit(None).cast("timestamp"))
    .withColumn("is_current", F.lit(True))
    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("unit_number"),
                F.col("calibration_timestamp").cast("string"),
                F.col("calibration_precision").cast("string")
            ),
            256
        )
    )
)

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

table_name = "dim_gps_unit"

# ---------------------------------------
# 1. Build source (same as before)
# ---------------------------------------
gps_src = (
    df.select(
        F.col("gps_unit").alias("unit_number"),
        F.col("gps_calibration_timestamp").alias("calibration_timestamp"),
        F.col("gps_calibration_precision").alias("calibration_precision")
    )
    .distinct()
    .withColumn("effective_start_ts", F.col("calibration_timestamp"))
    .withColumn("effective_end_ts", F.lit(None).cast("timestamp"))
    .withColumn("is_current", F.lit(True))
    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("unit_number"),
                F.col("calibration_timestamp").cast("string"),
                F.col("calibration_precision").cast("string")
            ),
            256
        )
    )
)

# ---------------------------------------
# 2. Check if table exists
# ---------------------------------------
if not spark.catalog.tableExists(table_name):

    # FIRST LOAD
    gps_dim = gps_src.withColumn("gps_unit_sk", F.monotonically_increasing_id())

    gps_dim.write.format("delta").mode("overwrite").saveAsTable(table_name)

else:

    # INCREMENTAL LOAD (SCD2)
    target = DeltaTable.forName(spark, table_name)

    current_dim = spark.table(table_name).filter("is_current = true")

    # ---------------------------------------
    # 3. Find changes (new or updated)
    # ---------------------------------------
    changes = (
        gps_src.alias("src")
        .join(current_dim.alias("tgt"), on="unit_number", how="left")
        .filter(
            F.col("tgt.unit_number").isNull() |
            (F.col("src.record_hash") != F.col("tgt.record_hash"))
        )
        .select("src.*")
    )

    # ---------------------------------------
    # 4. Expire old rows
    # ---------------------------------------
    target.alias("tgt").merge(
        changes.alias("src"),
        "tgt.unit_number = src.unit_number AND tgt.is_current = true"
    ).whenMatchedUpdate(
        set={
            "effective_end_ts": "src.effective_start_ts",
            "is_current": "false"
        }
    ).execute()

    # ---------------------------------------
    # 5. Insert new rows
    # ---------------------------------------
    new_rows = changes.withColumn("gps_unit_sk", F.monotonically_increasing_id())

    new_rows.write.format("delta").mode("append").saveAsTable(table_name)

In [0]:
gamma = df.select(
    F.col("gamma_detector_unit").alias("detector_unit_number"),
    F.col("gamma_calibration_timestamp").alias("calibration_timestamp"),
    F.col("gamma_calibration_precision").alias("calibration_precision")
)

cesium = df.select(
    F.col("cesium_detector_unit").alias("detector_unit_number"),
    F.col("cesium_calibration_timestamp").alias("calibration_timestamp"),
    F.col("cesium_calibration_precision").alias("calibration_precision")
)

thorium = df.select(
    F.col("thorium_detector_unit").alias("detector_unit_number"),
    F.col("thorium_calibration_timestamp").alias("calibration_timestamp"),
    F.col("thorium_calibration_precision").alias("calibration_precision")
)

detector_src = (
    gamma.union(cesium).union(thorium)
    .distinct()
    .withColumn("effective_start_ts", F.col("calibration_timestamp"))
    .withColumn("effective_end_ts", F.lit(None).cast("timestamp"))
    .withColumn("is_current", F.lit(True))
    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("detector_unit_number"),
                F.col("calibration_timestamp").cast("string"),
                F.col("calibration_precision").cast("string")
            ),
            256
        )
    )
)

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

table_name = "dim_detector"

# ---------------------------------------
# 1. Build detector source (union all)
# ---------------------------------------
gamma = df.select(
    F.col("gamma_detector_unit").alias("detector_unit_number"),
    F.col("gamma_calibration_timestamp").alias("calibration_timestamp"),
    F.col("gamma_calibration_precision").alias("calibration_precision")
)

cesium = df.select(
    F.col("cesium_detector_unit").alias("detector_unit_number"),
    F.col("cesium_calibration_timestamp").alias("calibration_timestamp"),
    F.col("cesium_calibration_precision").alias("calibration_precision")
)

thorium = df.select(
    F.col("thorium_detector_unit").alias("detector_unit_number"),
    F.col("thorium_calibration_timestamp").alias("calibration_timestamp"),
    F.col("thorium_calibration_precision").alias("calibration_precision")
)

detector_src = (
    gamma.union(cesium).union(thorium)
    .filter(F.col("detector_unit_number").isNotNull())
    .distinct()
    .withColumn("effective_start_ts", F.col("calibration_timestamp"))
    .withColumn("effective_end_ts", F.lit(None).cast("timestamp"))
    .withColumn("is_current", F.lit(True))
    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("detector_unit_number"),
                F.col("calibration_timestamp").cast("string"),
                F.col("calibration_precision").cast("string")
            ),
            256
        )
    )
)

# ---------------------------------------
# 2. First load OR incremental
# ---------------------------------------
if not spark.catalog.tableExists(table_name):

    # FIRST LOAD
    dim_detector = detector_src.withColumn(
        "detector_sk", F.monotonically_increasing_id()
    )

    dim_detector.write.format("delta").mode("overwrite").saveAsTable(table_name)

else:

    # INCREMENTAL (SCD2)
    target = DeltaTable.forName(spark, table_name)

    current_dim = spark.table(table_name).filter("is_current = true")

    # ---------------------------------------
    # 3. Detect new or changed records
    # ---------------------------------------
    changes = (
        detector_src.alias("src")
        .join(current_dim.alias("tgt"), on="detector_unit_number", how="left")
        .filter(
            F.col("tgt.detector_unit_number").isNull() |
            (F.col("src.record_hash") != F.col("tgt.record_hash"))
        )
        .select("src.*")
    )

    # ---------------------------------------
    # 4. Expire old rows
    # ---------------------------------------
    target.alias("tgt").merge(
        changes.alias("src"),
        "tgt.detector_unit_number = src.detector_unit_number AND tgt.is_current = true"
    ).whenMatchedUpdate(
        set={
            "effective_end_ts": "src.effective_start_ts",
            "is_current": "false"
        }
    ).execute()

    # ---------------------------------------
    # 5. Insert new rows
    # ---------------------------------------
    new_rows = changes.withColumn(
        "detector_sk", F.monotonically_increasing_id()
    )

    new_rows.write.format("delta").mode("append").saveAsTable(table_name)

In [0]:
display(spark.table("dim_time").orderBy("timestamp_id"))
display(spark.table("dim_location").orderBy("location_id"))
display(spark.table("dim_gps_unit").orderBy("unit_number"))
display(spark.table("dim_detector").orderBy("detector_unit_number"))
display(spark.table("dim_radiation_type").orderBy("radiation_type_id"))
display(spark.table("fact_radiation_measurement").orderBy("measurement_id"))
display(spark.table("fact_radiation_measurement").orderBy("measurement_id")) 